# Checkpoints_TMG — package original seed 42

Notebook này tải đích danh output `quangleealex/tmg-augmentation/versions/1`, xác minh artifact `seed=42`, `variant=original`, `augmentation=none`, rồi tạo hai output:

1. `tmg-toxic-nb03-seed42/seed_42/` — gắn trực tiếp output `Checkpoints_TMG` vào NB06.
2. `tmg-toxic-nb03-seed42.zip` — tải về để lưu trữ.

Không cần GPU; cần Internet cho `kagglehub` tải đúng Version 1. Không đóng gói recovery checkpoints vì Gradio chỉ cần inference artifact.


In [ ]:
# ===== USER SETTINGS — run this cell first =====
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # Packaging is CPU-only.

PACKAGE_NAME = 'tmg-toxic-nb03-seed42'
TARGET_SEED = 42
TARGET_VARIANT = 'original'
SOURCE_NOTEBOOK = 'quangleealex/tmg-augmentation'
SOURCE_VERSION = 1
DOWNLOAD_EXACT_VERSION = True
SOURCE_ORIGINAL_DIR = None  # Optional explicit /kaggle/input/.../original path.
print({'package': PACKAGE_NAME, 'seed': TARGET_SEED, 'variant': TARGET_VARIANT})


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import pandas as pd
import kagglehub
from zipfile import ZIP_STORED, ZipFile

INPUT_ROOT = Path('/kaggle/input')
WORKING_ROOT = Path('/kaggle/working')

if DOWNLOAD_EXACT_VERSION:
    version_handle = f'{SOURCE_NOTEBOOK}/versions/{SOURCE_VERSION}'
    downloaded_root = Path(kagglehub.notebook_output_download(version_handle))
    SEARCH_ROOT = downloaded_root
    print({'downloaded_notebook_version': version_handle,
           'downloaded_root': str(downloaded_root)})
else:
    SEARCH_ROOT = INPUT_ROOT

def read_json(path):
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except (OSError, json.JSONDecodeError):
        return None

# Resolve from the actual weight file because Kaggle may mount Notebook Outputs
# under a slug that differs from the visible notebook name.
if SOURCE_ORIGINAL_DIR is not None:
    source_dir_candidates = [Path(SOURCE_ORIGINAL_DIR)]
else:
    all_weight_paths = sorted(SEARCH_ROOT.rglob('model.safetensors'))
    print({'search_root': str(SEARCH_ROOT)})
    print({'mounted_inputs': [str(path) for path in INPUT_ROOT.iterdir()]})
    print({'discovered_model_weights': [str(path) for path in all_weight_paths]})
    source_dir_candidates = sorted({
        weight_path.parent.parent
        for weight_path in all_weight_paths
        if weight_path.parent.name == 'best_model' and
        weight_path.parent.parent.name == TARGET_VARIANT
    })

if len(source_dir_candidates) != 1:
    raise RuntimeError(
        'Expected exactly one model.safetensors under original/best_model; '
        f'found {[str(path) for path in source_dir_candidates]}. '
        'Use SOURCE_ORIGINAL_DIR to set the exact original folder if needed.'
    )

source_dir = source_dir_candidates[0]
source_model_dir = source_dir / 'best_model'
result_matches = sorted(source_dir.glob('validation_result*.json'))
prediction_matches = sorted(source_dir.glob('predictions_validation*.parquet'))
if len(result_matches) != 1 or len(prediction_matches) != 1:
    raise RuntimeError({
        'source_dir': str(source_dir),
        'validation_results': [str(path) for path in result_matches],
        'validation_predictions': [str(path) for path in prediction_matches],
    })
source_result_path = result_matches[0]
source_predictions_path = prediction_matches[0]
validation_result = read_json(source_result_path)
if not isinstance(validation_result, dict):
    raise ValueError(f'Invalid JSON object: {source_result_path}')

source_seed = validation_result.get('seed')
if source_seed is not None and int(source_seed) != TARGET_SEED:
    raise ValueError(f'Source metadata declares seed={source_seed}, not seed 42.')
source_variant = validation_result.get('variant')
if source_variant is not None and str(source_variant).lower() != TARGET_VARIANT:
    raise ValueError(f'Source metadata declares variant={source_variant!r}.')
source_augmentation = validation_result.get('augmentation')
if source_augmentation is not None and str(source_augmentation).lower() != 'none':
    raise ValueError(
        f'Source metadata declares augmentation={source_augmentation!r}.'
    )

required_model_files = (
    'config.json',
    'model.safetensors',
    'sentencepiece.bpe.model',
    'special_tokens_map.json',
    'tokenizer_config.json',
    'tokenizer.json',
)
for filename in required_model_files:
    path = source_model_dir / filename
    if not path.is_file():
        raise FileNotFoundError(path)

model_config = read_json(source_model_dir / 'config.json')
if model_config.get('model_type') != 'visobert-optimized-gate-v4':
    raise ValueError('Attached best_model is not Optimized GATED v4.')

raw_metrics = validation_result.get('raw_validation_metrics')
if not isinstance(raw_metrics, dict) or raw_metrics.get('macro_f1') is None:
    prediction_frame = pd.read_parquet(
        source_predictions_path
    )
    truth = prediction_frame['y_true'].astype(int)
    predicted = prediction_frame['y_pred_raw'].astype(int)
    per_class_f1 = []
    for label_id in (0, 1, 2):
        true_positive = int(((truth == label_id) & (predicted == label_id)).sum())
        false_positive = int(((truth != label_id) & (predicted == label_id)).sum())
        false_negative = int(((truth == label_id) & (predicted != label_id)).sum())
        denominator = 2 * true_positive + false_positive + false_negative
        per_class_f1.append(0.0 if denominator == 0 else
            2.0 * true_positive / denominator)
    raw_metrics = {'macro_f1': sum(per_class_f1) / len(per_class_f1)}

deployment_validation_result = dict(validation_result)
deployment_validation_result.update({
    'experiment': validation_result.get(
        'experiment', 'OPTIMIZED_GATE_V4_ORIGINAL_SEED42_DEPLOYMENT'),
    'seed': TARGET_SEED,
    'variant': TARGET_VARIANT,
    'augmentation': 'none',
    'raw_validation_metrics': raw_metrics,
    'packaging_note': (
        'Deployment metadata normalized from original/best_model. The unchanged '
        'source JSON is stored as source_validation_result.json.'
    ),
})

print({
    'source_dir': str(source_dir),
    'experiment': deployment_validation_result.get('experiment'),
    'source_seed': source_seed,
    'source_variant': source_variant,
    'source_augmentation': source_augmentation,
    'packaged_seed': deployment_validation_result.get('seed'),
    'packaged_variant': deployment_validation_result.get('variant'),
    'packaged_augmentation': deployment_validation_result.get('augmentation'),
    'model_type': model_config.get('model_type'),
})


In [ ]:
package_root = WORKING_ROOT / PACKAGE_NAME
seed_output_dir = package_root / 'seed_42'
model_output_dir = seed_output_dir / 'best_model'
zip_path = WORKING_ROOT / f'{PACKAGE_NAME}.zip'

if package_root.exists() or zip_path.exists():
    raise FileExistsError(
        'Package output already exists. Restart the Kaggle session before rebuilding '
        'so stale files cannot be mixed into the artifact.'
    )

model_output_dir.mkdir(parents=True, exist_ok=False)
for filename in required_model_files:
    shutil.copy2(source_model_dir / filename, model_output_dir / filename)
shutil.copy2(
    source_predictions_path,
    seed_output_dir / 'predictions_validation.parquet',
)
shutil.copy2(
    source_result_path, seed_output_dir / 'source_validation_result.json'
)
(seed_output_dir / 'validation_result.json').write_text(
    json.dumps(deployment_validation_result, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

artifact_files = sorted(path for path in seed_output_dir.rglob('*') if path.is_file())
artifact_manifest = {
    'artifact': PACKAGE_NAME,
    'schema_version': 1,
    'seed': TARGET_SEED,
    'variant': TARGET_VARIANT,
    'augmentation': 'none',
    'source_validation_result': str(source_result_path),
    'experiment': deployment_validation_result.get('experiment'),
    'model_type': model_config.get('model_type'),
    'raw_validation_macro_f1': deployment_validation_result[
        'raw_validation_metrics']['macro_f1'],
    'files': [{
        'path': path.relative_to(package_root).as_posix(),
        'size_bytes': path.stat().st_size,
        'sha256': sha256_file(path),
    } for path in artifact_files],
}
manifest_output_path = seed_output_dir / 'artifact_manifest.json'
manifest_output_path.write_text(
    json.dumps(artifact_manifest, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

with ZipFile(zip_path, mode='w', compression=ZIP_STORED, allowZip64=True) as archive:
    for path in sorted(item for item in package_root.rglob('*') if item.is_file()):
        archive.write(path, path.relative_to(package_root).as_posix())

print({
    'direct_nb06_input': str(seed_output_dir),
    'download_zip': str(zip_path),
    'zip_size_mb': round(zip_path.stat().st_size / 1024**2, 2),
    'packaged_files': len(artifact_files) + 1,
})


In [ ]:
# Final fail-closed verification before Save Version.
copied_result = read_json(seed_output_dir / 'validation_result.json')
copied_config = read_json(model_output_dir / 'config.json')
if copied_result.get('seed') != 42:
    raise ValueError('Packaged validation_result is not seed 42.')
if copied_result.get('variant') != 'original':
    raise ValueError('Packaged validation_result is not original.')
if copied_result.get('augmentation') != 'none':
    raise ValueError('Packaged artifact unexpectedly uses augmentation.')
if copied_config.get('model_type') != 'visobert-optimized-gate-v4':
    raise ValueError('Packaged config has the wrong model_type.')
if zip_path.stat().st_size < 300 * 1024**2:
    raise ValueError('ZIP is unexpectedly small; model weights may be missing.')

print('PACKAGE VERIFIED')
print('Next: Save Version -> Save & Run All.')
print('For NB06: Add Input -> Notebook Output Files -> Checkpoints_TMG.')
print('For backup: download tmg-toxic-nb03-seed42.zip from this notebook Output.')

from IPython.display import FileLink, display
display(FileLink(str(zip_path), result_html_prefix='Download checkpoint ZIP: '))
